# Rerank bằng cross-encoder

Bài 01 cho thấy hybrid thường **có** điều đúng trong top-50 nhưng không đặt nó ở hạng 1. Cross-encoder
`BAAI/bge-reranker-v2-m3` đọc cặp (câu hỏi, điều luật) **cùng lúc** nên chấm chính xác hơn bi-encoder, đổi lại mỗi
câu hỏi phải chạy model một lần cho mỗi ứng viên. Bạn viết hàm `rerank` tách khỏi model (test được bằng hàm điểm
giả), một hàm nạp cross-encoder **lười** (chỉ tải model khi chấm lần đầu), rồi đo MRR@10 và độ trễ trước/sau khi
rerank top-50 hybrid trên 100 câu đầu của tập ghim.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm, lớp. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật, chạy model, in bảng; test bỏ qua chúng.
- Cần tệp `data/raw/zalo-legal/cache/hybrid-top100.json` do bài 01 ghi ra: chạy bài 01 trước.
- Lần đầu tải model khoảng 2.3 GB từ Hugging Face. Rerank 100 câu × 50 ứng viên trên CPU mất khoảng 60 phút;
  điểm được cache nên chạy lại chỉ mất vài giây. Có GPU và PyTorch CUDA thì đổi `DEVICE = "cuda"`.
- Chấm: `uv run pytest tracks/ai-engineer/modules/13-rag/exercises/intermediate/03-reranking -q`

In [ ]:
# explore: tìm gốc repo, tải Zalo Legal nếu chưa có
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure

ZALO = ensure("zalo-legal")
CACHE = ZALO / "cache"
print(ZALO, sorted(p.name for p in ZALO.iterdir()))

In [ ]:
import json
import time
from dataclasses import dataclass
from typing import Callable

import numpy as np

## 1. `rerank` và cross-encoder nạp lười

`rerank(query, candidates, score_fn, top_n) -> list[Article]`:

- Gọi `score_fn(query, texts)` **đúng một lần** cho cả danh sách, với `texts[i] = f"{c.title}\n{c.text}"` của
  ứng viên thứ `i` (cross-encoder cần thấy tiêu đề điều). `score_fn` trả một điểm mỗi văn bản, điểm cao là liên quan
  hơn; có thể là `list[float]` hoặc mảng numpy.
- Trả `top_n` ứng viên điểm cao nhất, giảm dần. **Bằng điểm thì giữ thứ tự cũ** (thứ tự của retrieval): sort ổn định.
- Không có ứng viên thì trả `[]` và không gọi `score_fn`.

`cross_encoder_score_fn(model_name="BAAI/bge-reranker-v2-m3", device="cpu", max_length=512, batch_size=16)`
trả một hàm `score(query, texts) -> list[float]` dùng `sentence_transformers.CrossEncoder`:

- **Lười**: gọi `cross_encoder_score_fn(...)` không import `sentence_transformers`, không tải gì. Model được tạo
  (`CrossEncoder(model_name, device=device, max_length=max_length)`) ở lần chấm **đầu tiên**, rồi dùng lại cho mọi
  lần sau. Nhờ vậy định nghĩa hàm, import notebook và chạy test đều không tải 2.3 GB.
- Chấm bằng `model.predict([(query, t) for t in texts], batch_size=batch_size)`, trả `list[float]`.

`max_length=512` token: cross-encoder chỉ đọc phần đầu của điều dài (tiêu đề và vài khoản đầu).

In [ ]:
@dataclass
class Article:
    uid: str
    title: str
    text: str


def rerank(query: str, candidates: list[Article], score_fn: Callable[[str, list[str]], list[float]],
           top_n: int) -> list[Article]:
    """Chấm cả danh sách một lần bằng score_fn(query, ["title\ntext", ...]); top_n điểm cao nhất, bằng điểm giữ thứ tự cũ."""
    # TODO
    raise NotImplementedError


def cross_encoder_score_fn(model_name: str = "BAAI/bge-reranker-v2-m3", device: str = "cpu",
                           max_length: int = 512, batch_size: int = 16) -> Callable[[str, list[str]], list[float]]:
    """Hàm chấm dùng sentence_transformers.CrossEncoder, chỉ import và nạp model ở lần chấm đầu tiên."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: corpus, 100 câu đầu của tập ghim, top-100 hybrid của bài 01
corpus: dict[str, Article] = {}
with open(ZALO / "corpus.jsonl", encoding="utf-8") as f:
    for line in f:
        d = json.loads(line)
        corpus[d["_id"]] = Article(d["_id"], d["title"], d["text"])

qrels: dict[str, set[str]] = {}
with open(ZALO / "qrels" / "test.jsonl", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        qrels.setdefault(row["query-id"], set()).add(row["corpus-id"])
with open(ZALO / "queries.jsonl", encoding="utf-8") as f:
    query_text = {q["_id"]: q["text"] for q in map(json.loads, f)}

hybrid_path = CACHE / "hybrid-top100.json"
if not hybrid_path.exists():
    raise RuntimeError("Chưa có hybrid-top100.json: chạy hết notebook bài 01 trước.")
hybrid = json.loads(hybrid_path.read_text(encoding="utf-8"))
qids = list(qrels)[:100]
assert all(q in hybrid for q in qids), "hybrid-top100.json phải chứa 300 câu ghim của bài 01"


def mrr_at_10(ranked: list[list[str]], qs: list[str]) -> float:
    return float(np.mean([next((1 / r for r, u in enumerate(rk[:10], 1) if u in qrels[q]), 0.0)
                          for rk, q in zip(ranked, qs)]))


def recall_at(ranked: list[list[str]], qs: list[str], k: int) -> float:
    return float(np.mean([len(qrels[q] & set(rk[:k])) / len(qrels[q]) for rk, q in zip(ranked, qs)]))


before = [hybrid[q] for q in qids]
print(f"hybrid trên 100 câu: MRR@10 {mrr_at_10(before, qids):.3f}, R@1 {recall_at(before, qids, 1):.3f}, "
      f"R@50 (trần của rerank top-50) {recall_at(before, qids, 50):.3f}")

In [ ]:
# explore: thử trên một câu: nạp model (lần đầu tải 2.3 GB), xem thứ hạng trước và sau
DEVICE = "cpu"          # có PyTorch CUDA thì đổi thành "cuda"
score_fn = cross_encoder_score_fn(device=DEVICE)
q = qids[0]
cands = [corpus[u] for u in hybrid[q][:10]]
t0 = time.perf_counter()
reranked = rerank(query_text[q], cands, score_fn, top_n=5)
print(f"{query_text[q]} | đáp án {sorted(qrels[q])} | {time.perf_counter() - t0:.1f} s (gồm nạp model)")
print("trước:", [c.uid for c in cands[:5]])
print("sau:  ", [c.uid for c in reranked])

In [ ]:
# explore: rerank top-50 hybrid cho 100 câu, đo độ trễ; điểm được cache để chạy lại nhanh
TOP = 50
scores_path = CACHE / f"rerank-bge-reranker-v2-m3-top{TOP}.json"
cached = json.loads(scores_path.read_text(encoding="utf-8")) if scores_path.exists() else {}

after, latency = [], []
for n, q in enumerate(qids, start=1):
    cands = [corpus[u] for u in hybrid[q][:TOP]]
    if q in cached:          # dùng lại điểm đã chấm: vẫn đi qua rerank của bạn
        fn = lambda _query, _texts, s=cached[q]["scores"]: s
    else:
        def fn(query, texts, q=q):
            t0 = time.perf_counter()
            s = score_fn(query, texts)
            cached[q] = {"scores": s, "seconds": time.perf_counter() - t0}
            return s
    after.append([c.uid for c in rerank(query_text[q], cands, fn, top_n=TOP)])
    latency.append(cached[q]["seconds"])
    if n % 10 == 0:
        print(f"{n}/100 câu, trung vị {np.median(latency):.1f} s/câu", flush=True)
        _ = scores_path.write_text(json.dumps(cached), encoding="utf-8")
_ = scores_path.write_text(json.dumps(cached), encoding="utf-8")

In [ ]:
# explore: bảng trước/sau, câu lên hạng và tụt hạng, và rerank ít ứng viên hơn
import pandas as pd


def row(ranked):
    return {"MRR@10": mrr_at_10(ranked, qids), "R@1": recall_at(ranked, qids, 1),
            "R@5": recall_at(ranked, qids, 5), "R@10": recall_at(ranked, qids, 10)}


lat = np.array(latency)
table = {"Hybrid (bài 01)": row(before), f"Hybrid + rerank top-{TOP}": row(after)}
display(pd.DataFrame(table).T.round(3))
print(f"độ trễ rerank / câu ({DEVICE}): trung vị {np.median(lat):.1f} s, p95 {np.percentile(lat, 95):.1f} s, "
      f"tức {np.median(lat) / TOP * 1000:.0f} ms một cặp")


def first_hit(ranked: list[str], q: str) -> int:
    return next((r for r, u in enumerate(ranked, 1) if u in qrels[q]), 10**6)


moves = [first_hit(b, q) - first_hit(a, q) for b, a, q in zip(before, after, qids)]
print("điều đúng lên hạng:", sum(m > 0 for m in moves), "| tụt hạng:", sum(m < 0 for m in moves),
      "| giữ nguyên:", sum(m == 0 for m in moves))

# rerank top-K nhỏ hơn: dùng lại điểm đã có của K ứng viên đầu, chi phí tỉ lệ với K
for k in (10, 20, 50):
    ranked = []
    for q in qids:
        cands = [corpus[u] for u in hybrid[q][:k]]
        s = cached[q]["scores"][:k]
        ranked.append([c.uid for c in rerank(query_text[q], cands, lambda _q, _t, s=s: s, top_n=k)] + hybrid[q][k:])
    print(f"rerank top-{k:2d}: MRR@10 {mrr_at_10(ranked, qids):.3f}, R@1 {recall_at(ranked, qids, 1):.3f}, "
          f"ước độ trễ {np.median(lat) * k / TOP:.1f} s/câu")

## Đọc kết quả

So với bảng ở mục 4 lý thuyết (top-20 trên lát cắt 3 541 điều, MRR@10 0.773 lên 0.782), ở đây ứng viên đến từ
**toàn bộ** corpus và rerank top-50. Tự trả lời:

- Rerank cải thiện MRR@10 bao nhiêu, R@1 bao nhiêu? Bao nhiêu câu lên hạng, bao nhiêu câu tụt? Mức tăng có vượt
  nhiễu của 100 câu (mỗi câu đóng góp 0.01 vào R@1) không?
- Trên CPU, độ trễ rerank so với embed câu hỏi (vài chục ms) và BM25 (bài 01) thế nào? Với ngân sách 2 giây một
  câu, bạn chọn top-K bao nhiêu, hay cần GPU?
- Câu nào điều đúng nằm ngoài top-50 thì rerank không cứu được: trần là R@50 ở cell đầu.

Sau đó làm phần trắc nghiệm của bài trong app.